Config

In [24]:
import os
import mlflow


tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "mlruns")
mlflow.set_tracking_uri(tracking_uri)

print(f"MLFlow connecté à : {mlflow.get_tracking_uri()}")

MLFlow connecté à : https://user-avautrin-mlflow.user.lab.sspcloud.fr


Load model

In [25]:
MODEL_NAME = "model_KXCPI"

model_uri = f"models:/{MODEL_NAME}@production"
model = mlflow.lightgbm.load_model(model_uri)

print(f"Modèle chargé depuis : {model_uri}")
print(f"Type : {type(model)}")

Modèle chargé depuis : models:/model_KXCPI@production
Type : <class 'lightgbm.sklearn.LGBMRegressor'>


Kalshi API

In [26]:
import sys
sys.path.insert(0, "..")

from kalshi_predictor.series.data import fetch_tickers, fetch_data, build_features

SERIES_TICKER = "KXCPI"
WINDOW_DAYS = 60 
N_LAGS = 10

tickers = fetch_tickers(SERIES_TICKER)

17:24:45  INFO      Discovering tickers for series 'KXCPI'
17:24:45  INFO      Found 99 tickers in series 'KXCPI'


In [27]:
df_raw = fetch_data(SERIES_TICKER, tickers, window_days=WINDOW_DAYS)
df_raw.head()

17:24:45  INFO      Fetching candlesticks for 99 tickers …
Candlesticks: 100%|███████████████████████████████| 1/1 [00:01<00:00,  1.48s/it]
17:24:46  INFO      Fetching trade history in parallel (20 workers) …
Trades: 100%|███████████████████████████████████| 99/99 [00:06<00:00, 14.23it/s]
17:24:54  INFO      Panel ready: 93 tickers × 60 dates = 3576 rows


open  high   low  close    mean  \
ts                        ticker                                              
2026-02-09 00:00:00+00:00 KXCPI-26FEB-T0.0  0.00  0.00  0.00   0.00  0.0000   
                          KXCPI-26FEB-T0.1  0.00  0.00  0.00   0.00  0.0000   
                          KXCPI-26FEB-T0.2  0.75  0.75  0.75   0.75  0.7500   
                          KXCPI-26FEB-T0.3  0.02  0.61  0.02   0.61  0.0938   
                          KXCPI-26FEB-T0.4  0.39  0.39  0.39   0.39  0.3900   

                                            volume  price  vol_yes  vol_no  \
ts                        ticker                                             
2026-02-09 00:00:00+00:00 KXCPI-26FEB-T0.0     0.0   0.00     0.00    0.00   
                          KXCPI-26FEB-T0.1     0.0   0.00     0.00    0.00   
                          KXCPI-26FEB-T0.2     0.0  75.00    33.70   12.95   
                          KXCPI-26FEB-T0.3     0.0   9.38     3.61    0.80   
                          KXCPI-26FEB-T0.4     0.0  39.00     5.46    0.00   

                                            vol_total  
ts                        ticker                       
2026-02-09 00:00:00+00:00 KXCPI-26FEB-T0.0       0.00  
                          KXCPI-26FEB-T0.1       0.00  
                          KXCPI-26FEB-T0.2      46.65  
                          KXCPI-26FEB-T0.3       4.41  
                          KXCPI-26FEB-T0.4       5.46

feat

In [28]:
X, y = build_features(df_raw, n_lags=N_LAGS, min_obs=N_LAGS + 5)

17:24:54  INFO      Building features for 93 tickers …
Features: 100%|█████████████████████████████████| 93/93 [00:02<00:00, 39.52it/s]
17:24:56  INFO      Panel feature matrix: 33 rows × 50 features across 2 tickers


prediction

In [29]:
import pandas as pd
import numpy as np

# Récupérer la dernière date disponible
last_date = X.index.get_level_values("ts").max()
X_latest = X.loc[[last_date]]


pred

In [30]:
predictions = model.predict(X_latest)

results = pd.DataFrame(
    {
        "ticker": X_latest.index.get_level_values("ticker"),
        "predicted_return": predictions,
        "signal": np.sign(predictions),
    }
).sort_values("predicted_return", ascending=False)

results["signal"] = results["signal"].map({1.0: "LONG", -1.0: "SHORT", 0.0: "FLAT"})

results

,ticker,predicted_return,signal
0,KXCPI-26APR-T0.3,0.024030,LONG
1,KXCPI-26MAR-T0.5,0.004059,LONG
